# 09 — Exposure, netting, collateral and CVA

Counterparty risk is about the value of a **portfolio** at future dates. If the counterparty defaults at $t$, what is
lost is the positive part of the netted value of every trade under the same agreement, less the collateral held.

- $EPE(t) = E[\max(V(t), 0)]$: what the counterparty is expected to owe us.
- $ENE(t) = E[\min(V(t), 0)]$: what we are expected to owe them.
- $CVA = (1 - R)\sum_i EPE(t_i)\,\big(Q(t_{i-1}) - Q(t_i)\big)$, with $Q$ the counterparty's survival probability.

All exposures here are expectations of **deflated** values, so they are already discounted.

This notebook reads the tables behind `docs/RESULTS.md` section 11 (about 30 seconds the first time).

In [ ]:
import _util
from _util import run_analysis
import numpy as np, pandas as pd, matplotlib.pyplot as plt

t = run_analysis("xva")
sorted(t)

## The credit curve

Hazard rates, constant between CDS maturities, are bootstrapped so that each quoted spread makes its CDS worth zero.

**Hand check: the credit triangle.** To first order a CDS spread is the hazard rate times the loss given default,
$s \approx \lambda (1 - R)$. On an upward-sloping curve the *average* hazard to each maturity should be close to
$s/(1-R)$, while the hazard *on each interval* is higher than that at the long end.

In [ ]:
cr = t["xva_credit"]; cp = cr[cr.name == "counterparty"].reset_index(drop=True)
R = 0.4
cp["average hazard % (from survival)"] = -np.log(cp.survival_pct / 100) / cp.maturity * 100
cp["triangle s/(1-R) %"] = cp.spread_bp / 1e4 / (1 - R) * 100
cp[["maturity", "spread_bp", "hazard_pct", "survival_pct", "average hazard % (from survival)", "triangle s/(1-R) %"]].round(3)

## One swap: the engine against a closed form

For a single swap, $E[\max(V(t), 0)]$ on a coupon date is by definition the price of a European swaption on the
remaining swap. So the profile is a strip of swaptions with no simulation error, and the Monte Carlo engine must
reproduce it.

In [ ]:
sw = t["xva_swap"]
plt.plot(sw.time, sw.epe_closed_bp, "o-", label="EPE, closed form (swaption strip)")
plt.errorbar(sw.time, sw.epe_mc_bp, yerr=2 * sw.mc_se_bp, fmt="x", capsize=3, label="EPE, engine ± 2 s.e.")
plt.plot(sw.time, -sw.ene_closed_bp, "o-", label="−ENE, closed form")
plt.xlabel("years"); plt.ylabel("discounted, bp of notional"); plt.legend(); plt.show()
t["xva_swap_summary"]

**Hand check: CVA from the profile.** Rebuild the survival curve from the hazard rates, take the default probability
in each interval, weight the average exposure of the interval, and multiply by the loss given default. This should
reproduce the closed-form CVA in the table above.

In [ ]:
nodes, haz = cp.maturity.values, cp.hazard_pct.values / 100

def survival(x):
    # integral of a piecewise-constant hazard; the last rate continues beyond the last node
    edges = np.concatenate([[0.0], nodes[:-1], [np.inf]])
    return np.exp(-sum(h * np.clip(x - lo, 0.0, hi - lo) for h, lo, hi in zip(haz, edges[:-1], edges[1:])))

Q = np.array([survival(x) for x in sw.time])
cva = (1 - R) * np.sum(0.5 * (sw.epe_closed_bp.values[1:] + sw.epe_closed_bp.values[:-1]) * (Q[:-1] - Q[1:]))
print(f"by hand: {cva:.2f} bp     C++: {t['xva_swap_summary'].cva_bp[0]:.2f} bp")

## A netting set

Eight swaps with one counterparty. Without a netting agreement each trade is a separate claim, so the exposures add.
With one, only the net matters: $\max(\sum_i V_i, 0) \le \sum_i \max(V_i, 0)$.

In [ ]:
display(t["xva_book"])
p = t["xva_portfolio"]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.8))
a1.plot(p.time, p.gross_epe_bp, label="no netting: sum of the trades"); a1.plot(p.time, p.epe_bp, label="netted"); a1.plot(p.time, -p.ene_bp, label="netted, −ENE")
a1.set_title("expected exposure of the book"); a1.set_xlabel("years"); a1.set_ylabel("bp of notional"); a1.legend()
a2.plot(p.time, p.csa_threshold_bp, lw=1.2, label="threshold 50 bp"); a2.plot(p.time, p.csa_20d_bp, lw=1.2, label="zero threshold, 20-day margin period"); a2.plot(p.time, p.csa_zero_bp, lw=1.2, label="zero threshold, 10-day margin period")
a2.set_title("with collateral"); a2.set_xlabel("years"); a2.legend(); plt.show()
t["xva_table"]

## Why collateral does not remove exposure

After a default the surviving party needs the **margin period of risk** to notice, close out and re-hedge. During it
the book keeps moving and no more collateral arrives:

$$\text{exposure}(t) = \max\big(V(t) - C(t - \text{MPR}),\, 0\big).$$

Two things are visible in the right-hand plot, and both can be checked from the table.

**1. Square-root-of-time.** Away from coupon dates, what is left is a diffusion over the margin period, so doubling
the period should scale it by about $\sqrt{2}$.

In [ ]:
quiet = p[(p.coupon_date == 0) & (p.time > 0)]
ratio = quiet.csa_20d_bp / quiet.csa_zero_bp
print(f"20-day over 10-day residual exposure, away from coupon dates: mean {ratio.mean():.3f}, sqrt(2) = {np.sqrt(2):.3f}")

**2. Coupon spikes.** On a coupon date the book's value jumps by the net coupon, but the collateral held was called
on the value *before* the jump. Whoever was owed the coupon is exposed for it until the next call. These spikes do
not scale with the margin period, which is why the CVA ratio between the two agreements is below $\sqrt{2}$.

In [ ]:
spike = p[p.coupon_date == 1]
print(f"zero-threshold residual exposure: {quiet.csa_zero_bp.mean():.1f} bp away from coupon dates, {spike.csa_zero_bp.mean():.1f} bp on them")
tab = t["xva_table"].set_index("case")
print(f"CVA, 20-day over 10-day margin period: {tab.cva_bp['zero threshold; 20-day margin period'] / tab.cva_bp['zero threshold; 10-day margin period']:.2f}")

Points to be able to make:

- Netting is the largest single mitigant here; collateral with a zero threshold cuts what is left by more than ten
  times again, and still not to zero.
- A threshold is uncollateralised exposure by design.
- A minimum transfer amount barely changes the result: a call that is not made leaves a small value uncollateralised
  in both directions.
- Simplifications: a floating coupon in progress is valued at the term rate fixed at its period start (exact on
  coupon dates); no initial margin; default and exposure are independent in this notebook (notebook 10 relaxes that).